# Contextual Compression for RAG

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cyruslayo/castalia/blob/main/rag/contextual_compression.ipynb)

## Overview

In a standard RAG pipeline, a retriever returns the **top-k most similar chunks** to the user's query. But chunks are blunt instruments — each one was cut at a fixed character boundary, so a 500-token chunk that *contains* the answer usually also contains 300+ tokens of irrelevant filler: background context, adjacent sentences about different subtopics, and transition phrases. When this bloated context is pasted into the LLM prompt, two things go wrong:

1. **Token waste** — the model's finite context window fills up with noise, leaving less room for truly helpful information.
2. **Distraction** — research shows that LLMs perform worse when relevant facts are buried inside long, irrelevant passages ("lost in the middle" effect).

**Contextual compression** is a post-retrieval step that *compresses* each retrieved chunk down to only the parts that are relevant to the current query. This notebook builds the entire mechanism from scratch — no LangChain, no LlamaIndex — using a local **Qwen3-8B** model for abstractive compression and **BGE embeddings** for extractive compression.

## What You'll Learn

| Section | Technique |
|---|---|
| §1 | The context bloat problem — visualised with a real example |
| §2 | Extractive compression — sentence-level cosine filtering |
| §3 | Abstractive compression — query-focused LLM summarisation |
| §4 | Information density analysis — compression ratios and token savings |
| §5 | Impact on generation — side-by-side answer comparison |
| §6 | Complete pipeline — Retrieve → Compress → Generate |
| §7 | Synthesis — when (and when not) to use compression |

In [ ]:
# Install all required packages (Colab)
!pip install -q transformers>=4.51.0 accelerate bitsandbytes torch
!pip install -q sentence-transformers faiss-cpu numpy

## Setup — Load Qwen3-8B (4-bit) and BGE Embeddings

We load the **Qwen/Qwen3-8B** model with 4-bit NF4 quantisation so it fits on a free Colab T4 GPU (~8 GB VRAM). Model weights are cached to Google Drive to avoid re-downloading each session.

In [ ]:
import torch
from google.colab import drive
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

drive.mount('/content/drive')
CACHE_DIR = "/content/drive/MyDrive/models"
MODEL_NAME = "Qwen/Qwen3-8B"

quantization_config = BitsAndBytesConfig(
    load_in_4bit=True, bnb_4bit_compute_dtype=torch.bfloat16, bnb_4bit_quant_type="nf4",
)
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, cache_dir=CACHE_DIR)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME, device_map="auto", quantization_config=quantization_config,
    cache_dir=CACHE_DIR, torch_dtype="auto",
)

def generate(messages, max_new_tokens=1024, temperature=0.6, do_sample=True, thinking=True):
    """Generate a response, optionally with Qwen3's native thinking mode.
    Returns (thinking_text, answer) when thinking=True, otherwise just the answer string."""
    text = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True,
        enable_thinking=thinking,
    )
    inputs = tokenizer([text], return_tensors="pt").to(model.device)
    gen_kwargs = dict(
        max_new_tokens=max_new_tokens, do_sample=do_sample, top_k=20,
        temperature=temperature if do_sample else None,
        top_p=0.95 if thinking else 0.8,
    )
    output_ids = model.generate(**inputs, **gen_kwargs)
    token_ids = output_ids[0][inputs.input_ids.shape[1]:].tolist()
    if thinking:
        try:
            idx = len(token_ids) - token_ids[::-1].index(151668)
        except ValueError:
            idx = 0
        thought = tokenizer.decode(token_ids[:idx], skip_special_tokens=True).strip()
        answer  = tokenizer.decode(token_ids[idx:], skip_special_tokens=True).strip()
        return thought, answer
    return tokenizer.decode(token_ids, skip_special_tokens=True).strip()

print(f"✓ Qwen3-8B loaded — device: {model.device}")

### Load BGE Embeddings

**BAAI/bge-base-en-v1.5** produces 768-dimensional embeddings that we'll use for both the vector store and sentence-level relevance scoring.

In [ ]:
from sentence_transformers import SentenceTransformer
import numpy as np

embed_model = SentenceTransformer("BAAI/bge-base-en-v1.5", cache_folder=CACHE_DIR)

def embed_texts(texts):
    """Embed a list of strings; returns a numpy array of shape (n, 768)."""
    return embed_model.encode(texts, normalize_embeddings=True, show_progress_bar=False)

# Quick sanity check
test_emb = embed_texts(["hello world"])
print(f"✓ BGE loaded — embedding dim: {test_emb.shape[1]}")

## §1 — The Context Bloat Problem

### Synthetic Knowledge Base

We create a small but realistic knowledge base about renewable energy. Each "document" is a multi-sentence paragraph — exactly the kind of chunk a RAG system would retrieve. Notice how each chunk mixes relevant and irrelevant information relative to any particular query.

In [ ]:
KNOWLEDGE_BASE = [
    (
        "Solar energy is one of the fastest-growing sources of electricity worldwide. "
        "Photovoltaic (PV) panels convert sunlight directly into electricity using semiconductor materials, "
        "most commonly silicon. The efficiency of commercial PV panels ranges from 15% to 22%, "
        "though laboratory cells have exceeded 47% under concentrated sunlight. "
        "Solar farms require significant land area — roughly 5 to 10 acres per megawatt of capacity. "
        "The cost of solar electricity has dropped by over 89% since 2010, making it cheaper than coal "
        "in many regions. However, solar output is intermittent: panels generate no power at night and "
        "reduced power on cloudy days, which creates challenges for grid operators. "
        "China is the world's largest manufacturer of solar panels, producing over 80% of global supply."
    ),
    (
        "Wind turbines convert kinetic energy from moving air into electricity via a generator. "
        "Modern onshore turbines have capacities of 2 to 5 megawatts, while offshore turbines can "
        "exceed 15 megawatts. The global installed wind capacity surpassed 900 GW in 2023. "
        "Wind energy's capacity factor — the ratio of actual output to maximum possible output — "
        "typically ranges from 25% to 45%, depending on location and turbine design. "
        "Bird and bat mortality is an environmental concern, with estimates of 100,000 to 500,000 bird "
        "deaths per year in the United States alone from turbine collisions. "
        "Denmark generates over 50% of its electricity from wind, the highest share of any country. "
        "Noise pollution from turbines has led to setback regulations requiring minimum distances "
        "from residential areas, typically 500 to 1,000 metres."
    ),
    (
        "Battery storage is critical for integrating intermittent renewable sources into the grid. "
        "Lithium-ion batteries dominate the market, accounting for over 90% of new grid-scale "
        "installations. Their energy density ranges from 150 to 265 Wh/kg, and cycle life typically "
        "exceeds 3,000 full charge-discharge cycles. The cost of lithium-ion battery packs has fallen "
        "from $1,200/kWh in 2010 to approximately $140/kWh in 2023. "
        "Alternative technologies include flow batteries, which offer longer duration storage (4-12 hours) "
        "but lower energy density, and sodium-ion batteries, which avoid scarce lithium. "
        "Grid-scale batteries help with peak shaving, frequency regulation, and renewable energy "
        "time-shifting. Tesla's Hornsdale Power Reserve in Australia was one of the first large-scale "
        "lithium-ion installations, with a capacity of 150 MW / 194 MWh."
    ),
    (
        "Hydrogen produced via electrolysis using renewable electricity is called green hydrogen. "
        "The process splits water into hydrogen and oxygen, requiring about 50-55 kWh of electricity "
        "per kilogram of hydrogen. Current electrolyser efficiency ranges from 60% to 80%. "
        "Green hydrogen can be used as a fuel for heavy transport, industrial heating, and long-term "
        "energy storage. The cost of green hydrogen is currently $4-8 per kilogram, compared to "
        "$1-2 for grey hydrogen produced from natural gas. "
        "The European Union aims to produce 10 million tonnes of green hydrogen annually by 2030. "
        "Storing hydrogen is challenging due to its low volumetric energy density — it must be "
        "compressed to 350-700 bar or liquefied at -253°C. "
        "Fuel cell vehicles convert hydrogen back to electricity with about 60% efficiency."
    ),
    (
        "Nuclear energy provides about 10% of global electricity from approximately 440 reactors "
        "in 32 countries. Unlike renewables, nuclear plants generate baseload power continuously, "
        "with capacity factors exceeding 90%. A single uranium fuel pellet contains as much energy "
        "as one tonne of coal. Small modular reactors (SMRs) are a new design category with capacities "
        "under 300 MW, intended to be factory-built and transported to site. "
        "Nuclear waste remains radioactive for thousands of years and requires deep geological disposal. "
        "France generates about 70% of its electricity from nuclear power, the highest share globally. "
        "The levelized cost of electricity from new nuclear plants ranges from $112 to $189 per MWh, "
        "making it one of the more expensive low-carbon options. Construction times for new plants "
        "average 7 to 10 years, with frequent cost overruns."
    ),
    (
        "Carbon capture and storage (CCS) involves capturing CO₂ emissions from industrial processes "
        "or power plants and storing them underground in geological formations. "
        "Post-combustion capture uses chemical solvents (typically amines) to absorb CO₂ from flue gas. "
        "Pre-combustion capture converts fuel to hydrogen and CO₂ before combustion. "
        "Direct air capture (DAC) removes CO₂ directly from the atmosphere but requires 6-10 times "
        "more energy than point-source capture. The global CCS capacity was approximately 45 million "
        "tonnes of CO₂ per year in 2023, far below the gigatonne-scale needed for climate targets. "
        "The cost of CCS ranges from $50-120 per tonne of CO₂ for point-source capture, "
        "and $250-600 per tonne for direct air capture. Norway's Sleipner project, operational since "
        "1996, was the world's first commercial CCS installation."
    ),
]

print(f"Knowledge base: {len(KNOWLEDGE_BASE)} chunks")
for i, chunk in enumerate(KNOWLEDGE_BASE):
    print(f"  Chunk {i}: {len(chunk.split())} words, {len(chunk)} chars")

### Build a FAISS Vector Index

We embed every chunk and build a FAISS inner-product index for retrieval. Since BGE embeddings are L2-normalised, inner product equals cosine similarity.

In [ ]:
import faiss

chunk_embeddings = embed_texts(KNOWLEDGE_BASE)
dimension = chunk_embeddings.shape[1]
index = faiss.IndexFlatIP(dimension)  # inner product = cosine for normalised vecs
index.add(chunk_embeddings.astype(np.float32))

def retrieve(query, k=3):
    """Return top-k (score, chunk_index, chunk_text) tuples."""
    q_emb = embed_texts([query]).astype(np.float32)
    scores, indices = index.search(q_emb, k)
    results = []
    for score, idx in zip(scores[0], indices[0]):
        results.append((float(score), int(idx), KNOWLEDGE_BASE[idx]))
    return results

print(f"✓ FAISS index built — {index.ntotal} vectors, dim={dimension}")

### Visualising the Context Bloat Problem

Let's retrieve chunks for a specific query and highlight which sentences are actually relevant. This makes the "bloat" tangible.

In [ ]:
import re

def split_sentences(text):
    """Split text into sentences using a simple regex."""
    return [s.strip() for s in re.split(r'(?<=[.!?])\s+', text) if s.strip()]

query = "What is the cost of battery storage and how has it changed?"
results = retrieve(query, k=3)

print(f"Query: {query!r}\n")
for rank, (score, idx, chunk) in enumerate(results):
    sentences = split_sentences(chunk)
    print(f"--- Chunk {idx} (rank #{rank+1}, score={score:.4f}) ---")
    # Score each sentence's relevance to the query via embedding similarity
    q_emb = embed_texts([query])
    s_embs = embed_texts(sentences)
    sims = (s_embs @ q_emb.T).flatten()
    for sent, sim in zip(sentences, sims):
        marker = "✅" if sim > 0.65 else "❌"
        print(f"  {marker} [{sim:.3f}] {sent}")
    relevant = sum(1 for s in sims if s > 0.65)
    print(f"  → {relevant}/{len(sentences)} sentences relevant ({relevant/len(sentences)*100:.0f}%)\n")

## §2 — What Is Contextual Compression?

The example above is typical: **only 30-50% of the sentences** in a retrieved chunk are actually relevant to the query. The rest is noise.

**Contextual compression** is a post-retrieval processing step that sits between the retriever and the generator:

```
Query → Retriever → [Chunk₁, Chunk₂, ...] → Compressor → [Compressed₁, Compressed₂, ...] → Generator → Answer
```

There are two families of compression:

| Method | How it works | Pros | Cons |
|---|---|---|---|
| **Extractive** | Select only relevant sentences from the chunk (discard the rest) | Fast, no hallucination risk, preserves exact wording | Can't synthesise or rephrase; may keep slightly-too-much |
| **Abstractive** | Use an LLM to summarise/rewrite the chunk w.r.t. the query | Maximum compression, can merge info | Slower (LLM call per chunk), small hallucination risk |

Both methods share the same goal: **increase information density** — more relevant tokens per total token.

## §3 — Extractive Compression

### Algorithm

1. Split the chunk into individual sentences
2. Embed the query and each sentence with BGE
3. Compute cosine similarity between query and each sentence
4. Keep only sentences whose similarity exceeds a threshold

This is conceptually simple but remarkably effective. It's analogous to how a human would highlight relevant lines in a textbook.

In [ ]:
def extractive_compress(chunk, query, threshold=0.55):
    """
    Compress a chunk by keeping only sentences whose embedding similarity
    to the query exceeds the threshold.

    Returns:
        compressed_text: str — the surviving sentences joined together
        stats: dict — metadata about what was kept/removed
    """
    sentences = split_sentences(chunk)
    if not sentences:
        return chunk, {"original_sents": 0, "kept_sents": 0}

    q_emb = embed_texts([query])
    s_embs = embed_texts(sentences)
    similarities = (s_embs @ q_emb.T).flatten()

    kept = []
    dropped = []
    for sent, sim in zip(sentences, similarities):
        if sim >= threshold:
            kept.append((sent, float(sim)))
        else:
            dropped.append((sent, float(sim)))

    compressed = " ".join(s for s, _ in kept) if kept else sentences[0]  # fallback: keep best

    stats = {
        "original_sents": len(sentences),
        "kept_sents": len(kept),
        "dropped_sents": len(dropped),
        "original_chars": len(chunk),
        "compressed_chars": len(compressed),
        "compression_ratio": len(compressed) / len(chunk) if chunk else 1.0,
        "kept_details": kept,
        "dropped_details": dropped,
    }
    return compressed, stats

print("✓ extractive_compress() defined")

### Extractive Compression in Action

Let's apply extractive compression to each retrieved chunk and see what survives.

In [ ]:
query = "What is the cost of battery storage and how has it changed?"
results = retrieve(query, k=3)

print(f"Query: {query!r}\n")
for rank, (score, idx, chunk) in enumerate(results):
    compressed, stats = extractive_compress(chunk, query, threshold=0.55)
    print(f"--- Chunk {idx} (rank #{rank+1}) ---")
    print(f"Original: {stats['original_sents']} sentences, {stats['original_chars']} chars")
    print(f"Compressed: {stats['kept_sents']} sentences, {stats['compressed_chars']} chars")
    print(f"Compression ratio: {stats['compression_ratio']:.1%}")
    print(f"\nKept:")
    for sent, sim in stats['kept_details']:
        print(f"  ✅ [{sim:.3f}] {sent}")
    print(f"Dropped:")
    for sent, sim in stats['dropped_details']:
        print(f"  ❌ [{sim:.3f}] {sent}")
    print()

### Threshold Sensitivity

The threshold is the key hyperparameter. Too high → we drop relevant sentences. Too low → we keep noise. Let's visualise the trade-off.

In [ ]:
query = "What is the cost of battery storage and how has it changed?"
chunk = KNOWLEDGE_BASE[2]  # battery storage chunk

thresholds = [0.40, 0.45, 0.50, 0.55, 0.60, 0.65, 0.70, 0.75]
print(f"Query: {query!r}")
print(f"Chunk: battery storage ({len(split_sentences(chunk))} sentences)\n")
print(f"{'Threshold':>10} {'Kept':>6} {'Dropped':>8} {'Ratio':>8} {'Compressed text (first 100 chars)'}")
print("-" * 90)
for t in thresholds:
    comp, stats = extractive_compress(chunk, query, threshold=t)
    print(f"{t:>10.2f} {stats['kept_sents']:>6} {stats['dropped_sents']:>8} {stats['compression_ratio']:>8.1%}   {comp[:100]}...")

## §4 — Abstractive Compression

Extractive compression keeps original sentences verbatim. **Abstractive compression** goes further: it uses the LLM to rewrite the chunk, extracting and synthesising only the query-relevant information into a concise passage.

### Advantages over Extractive
- Can **merge information** from multiple sentences into one
- Achieves **higher compression ratios** (often 60-80% reduction)
- Can **rephrase** for clarity

### Risks
- **Latency**: requires an LLM call per chunk (can be batched)
- **Hallucination**: the LLM might add information not in the original chunk
- **Cost**: if using an API model, compression costs money

In [ ]:
COMPRESSION_PROMPT = """You are a precise information extractor. Given a QUERY and a DOCUMENT CHUNK, extract ONLY the information from the chunk that is directly relevant to answering the query.

Rules:
- Include ONLY facts, figures, and statements that help answer the query
- Do NOT add any information not present in the chunk
- Do NOT include introductory phrases like "The chunk mentions..."
- If nothing in the chunk is relevant, respond with "NO_RELEVANT_INFO"
- Be concise but preserve important details (numbers, dates, names)

QUERY: {query}

DOCUMENT CHUNK:
{chunk}

RELEVANT INFORMATION:"""

def abstractive_compress(chunk, query):
    """
    Use the LLM to extract only query-relevant information from a chunk.
    Returns (compressed_text, stats_dict).
    """
    prompt = COMPRESSION_PROMPT.format(query=query, chunk=chunk)
    messages = [{"role": "user", "content": prompt}]
    answer = generate(messages, max_new_tokens=512, temperature=0.1, thinking=False)
    compressed = answer.strip()
    if compressed == "NO_RELEVANT_INFO":
        compressed = ""
    stats = {
        "original_chars": len(chunk),
        "compressed_chars": len(compressed),
        "compression_ratio": len(compressed) / len(chunk) if chunk else 1.0,
        "original_tokens": len(tokenizer.encode(chunk)),
        "compressed_tokens": len(tokenizer.encode(compressed)) if compressed else 0,
    }
    return compressed, stats

print("✓ abstractive_compress() defined")

### Abstractive Compression in Action

Let's compress the same chunks we retrieved earlier, this time using the LLM.

In [ ]:
query = "What is the cost of battery storage and how has it changed?"
results = retrieve(query, k=3)

abstractive_results = []
print(f"Query: {query!r}\n")
for rank, (score, idx, chunk) in enumerate(results):
    compressed, stats = abstractive_compress(chunk, query)
    abstractive_results.append((idx, chunk, compressed, stats))
    print(f"--- Chunk {idx} (rank #{rank+1}) ---")
    print(f"Original ({stats['original_tokens']} tokens, {stats['original_chars']} chars):")
    print(f"  {chunk[:150]}...")
    print(f"Compressed ({stats['compressed_tokens']} tokens, {stats['compressed_chars']} chars):")
    print(f"  {compressed}")
    print(f"Compression ratio: {stats['compression_ratio']:.1%}")
    print()

### Extractive vs. Abstractive — Side by Side

Let's compare both methods on the same chunk to see how they differ in output and compression.

In [ ]:
query = "What is the cost of battery storage and how has it changed?"
chunk = KNOWLEDGE_BASE[2]  # battery chunk

ext_compressed, ext_stats = extractive_compress(chunk, query, threshold=0.55)
abs_compressed, abs_stats = abstractive_compress(chunk, query)

print(f"Query: {query!r}\n")
print(f"{'Method':<14} {'Orig chars':>12} {'Comp chars':>12} {'Ratio':>8} {'Orig tokens':>12} {'Comp tokens':>12}")
print("-" * 80)
print(f"{'Extractive':<14} {ext_stats['original_chars']:>12} {ext_stats['compressed_chars']:>12} "
      f"{ext_stats['compression_ratio']:>8.1%} {len(tokenizer.encode(chunk)):>12} {len(tokenizer.encode(ext_compressed)):>12}")
print(f"{'Abstractive':<14} {abs_stats['original_chars']:>12} {abs_stats['compressed_chars']:>12} "
      f"{abs_stats['compression_ratio']:>8.1%} {abs_stats['original_tokens']:>12} {abs_stats['compressed_tokens']:>12}")
print(f"\n--- Extractive output ---\n{ext_compressed}")
print(f"\n--- Abstractive output ---\n{abs_compressed}")

## §5 — Information Density Analysis

**Information density** measures how much of the text is actually useful for answering the query. We define it as:

$$\text{Density} = \frac{\text{relevant tokens}}{\text{total tokens}}$$

For extractive compression, "relevant tokens" are the tokens in kept sentences. For the original uncompressed chunk, we estimate relevant tokens by counting tokens in sentences with similarity > threshold.

A higher density means less wasted context window — which directly translates to better LLM performance.

In [ ]:
queries = [
    "What is the cost of battery storage and how has it changed?",
    "How does wind energy affect wildlife?",
    "What is green hydrogen and how is it produced?",
    "What are the challenges of nuclear waste disposal?",
    "How does direct air capture work and what does it cost?",
]

print(f"{'Query (truncated)':<50} {'Method':<14} {'Tokens':>8} {'Density':>8}")
print("=" * 90)

for query in queries:
    results = retrieve(query, k=2)
    # Uncompressed: concatenate raw chunks
    raw_context = " ".join(chunk for _, _, chunk in results)
    raw_tokens = len(tokenizer.encode(raw_context))
    # Estimate relevant tokens in raw context
    raw_sents = split_sentences(raw_context)
    q_emb = embed_texts([query])
    s_embs = embed_texts(raw_sents)
    sims = (s_embs @ q_emb.T).flatten()
    relevant_raw = sum(len(tokenizer.encode(s)) for s, sim in zip(raw_sents, sims) if sim > 0.55)
    raw_density = relevant_raw / raw_tokens if raw_tokens else 0

    # Extractive
    ext_parts = []
    for _, _, chunk in results:
        comp, _ = extractive_compress(chunk, query, threshold=0.55)
        ext_parts.append(comp)
    ext_context = " ".join(ext_parts)
    ext_tokens = len(tokenizer.encode(ext_context))

    # Abstractive
    abs_parts = []
    for _, _, chunk in results:
        comp, _ = abstractive_compress(chunk, query)
        if comp:
            abs_parts.append(comp)
    abs_context = " ".join(abs_parts) if abs_parts else "(empty)"
    abs_tokens = len(tokenizer.encode(abs_context))

    q_short = query[:48] + ".." if len(query) > 48 else query
    print(f"{q_short:<50} {'Uncompressed':<14} {raw_tokens:>8} {raw_density:>8.1%}")
    print(f"{'':<50} {'Extractive':<14} {ext_tokens:>8} {'~100%':>8}")
    print(f"{'':<50} {'Abstractive':<14} {abs_tokens:>8} {'~100%':>8}")
    print(f"{'':<50} {'Savings':<14} {raw_tokens - abs_tokens:>8} {'tokens saved':>8}")
    print()

## §6 — Impact on Generation Quality

The real test: does compression actually improve the LLM's answers? We compare three approaches:

1. **Uncompressed** — raw retrieved chunks pasted into the prompt
2. **Extractive** — only relevant sentences kept
3. **Abstractive** — LLM-compressed context

We'll use a consistent prompt template and compare the answers side-by-side.

In [ ]:
RAG_PROMPT = """Answer the following question using ONLY the provided context. If the context doesn't contain enough information, say so.

Context:
{context}

Question: {question}

Answer:"""

def rag_answer(question, context):
    """Generate a RAG answer using the given context."""
    prompt = RAG_PROMPT.format(context=context, question=question)
    messages = [{"role": "user", "content": prompt}]
    answer = generate(messages, max_new_tokens=512, temperature=0.3, thinking=False)
    return answer

print("✓ rag_answer() defined")

In [ ]:
query = "What is the cost of battery storage and how has it changed over time?"
results = retrieve(query, k=3)

# --- Uncompressed context ---
raw_context = "\n\n".join(chunk for _, _, chunk in results)
raw_tokens = len(tokenizer.encode(raw_context))

# --- Extractive context ---
ext_parts = []
for _, _, chunk in results:
    comp, _ = extractive_compress(chunk, query, threshold=0.55)
    ext_parts.append(comp)
ext_context = "\n\n".join(ext_parts)
ext_tokens = len(tokenizer.encode(ext_context))

# --- Abstractive context ---
abs_parts = []
for _, _, chunk in results:
    comp, _ = abstractive_compress(chunk, query)
    if comp:
        abs_parts.append(comp)
abs_context = "\n\n".join(abs_parts)
abs_tokens = len(tokenizer.encode(abs_context))

print(f"Query: {query!r}")
print(f"Context tokens — Raw: {raw_tokens}, Extractive: {ext_tokens}, Abstractive: {abs_tokens}\n")

# Generate answers
print("=" * 70)
print("UNCOMPRESSED ANSWER")
print("=" * 70)
raw_answer = rag_answer(query, raw_context)
print(raw_answer)

print("\n" + "=" * 70)
print("EXTRACTIVE COMPRESSION ANSWER")
print("=" * 70)
ext_answer = rag_answer(query, ext_context)
print(ext_answer)

print("\n" + "=" * 70)
print("ABSTRACTIVE COMPRESSION ANSWER")
print("=" * 70)
abs_answer = rag_answer(query, abs_context)
print(abs_answer)

### Second Example — Different Query

Let's try a query where the relevant information is spread thinly across chunks with lots of irrelevant material.

In [ ]:
query = "What are the environmental concerns associated with wind turbines?"
results = retrieve(query, k=3)

raw_context = "\n\n".join(chunk for _, _, chunk in results)
raw_tokens = len(tokenizer.encode(raw_context))

ext_parts = [extractive_compress(chunk, query, threshold=0.50)[0] for _, _, chunk in results]
ext_context = "\n\n".join(ext_parts)
ext_tokens = len(tokenizer.encode(ext_context))

abs_parts = [c for c, _ in (abstractive_compress(chunk, query) for _, _, chunk in results) if c]
abs_context = "\n\n".join(abs_parts)
abs_tokens = len(tokenizer.encode(abs_context))

print(f"Query: {query!r}")
print(f"Context tokens — Raw: {raw_tokens}, Extractive: {ext_tokens}, Abstractive: {abs_tokens}\n")

print("=" * 70)
print("UNCOMPRESSED ANSWER")
print("=" * 70)
print(rag_answer(query, raw_context))

print("\n" + "=" * 70)
print("EXTRACTIVE COMPRESSION ANSWER")
print("=" * 70)
print(rag_answer(query, ext_context))

print("\n" + "=" * 70)
print("ABSTRACTIVE COMPRESSION ANSWER")
print("=" * 70)
print(rag_answer(query, abs_context))

## §7 — Complete Pipeline: Retrieve → Compress → Generate

Now let's package everything into a clean, reusable pipeline class that supports both compression modes.

In [ ]:
class CompressedRAGPipeline:
    """
    A complete RAG pipeline with optional contextual compression.
    Modes: 'none', 'extractive', 'abstractive'
    """

    def __init__(self, knowledge_base, embed_fn, index, compress_mode="extractive",
                 extractive_threshold=0.55, top_k=3):
        self.kb = knowledge_base
        self.embed_fn = embed_fn
        self.index = index
        self.mode = compress_mode
        self.threshold = extractive_threshold
        self.top_k = top_k

    def retrieve(self, query):
        q_emb = self.embed_fn([query]).astype(np.float32)
        scores, indices = self.index.search(q_emb, self.top_k)
        return [(float(s), int(i), self.kb[i]) for s, i in zip(scores[0], indices[0])]

    def compress(self, chunks, query):
        compressed = []
        stats_list = []
        for chunk in chunks:
            if self.mode == "extractive":
                comp, stats = extractive_compress(chunk, query, self.threshold)
            elif self.mode == "abstractive":
                comp, stats = abstractive_compress(chunk, query)
            else:  # no compression
                comp = chunk
                stats = {"original_chars": len(chunk), "compressed_chars": len(chunk),
                         "compression_ratio": 1.0}
            compressed.append(comp)
            stats_list.append(stats)
        return compressed, stats_list

    def generate_answer(self, query, context):
        return rag_answer(query, context)

    def run(self, query, verbose=True):
        # Step 1: Retrieve
        results = self.retrieve(query)
        raw_chunks = [chunk for _, _, chunk in results]
        if verbose:
            print(f"[1] Retrieved {len(results)} chunks")

        # Step 2: Compress
        compressed_chunks, stats_list = self.compress(raw_chunks, query)
        if verbose:
            total_orig = sum(s['original_chars'] for s in stats_list)
            total_comp = sum(s['compressed_chars'] for s in stats_list)
            print(f"[2] Compressed ({self.mode}): {total_orig} → {total_comp} chars "
                  f"({total_comp/total_orig:.1%})")

        # Step 3: Generate
        context = "\n\n".join(c for c in compressed_chunks if c)
        answer = self.generate_answer(query, context)
        if verbose:
            print(f"[3] Generated answer ({len(tokenizer.encode(answer))} tokens)")
        return answer

print("✓ CompressedRAGPipeline class defined")

### Running the Complete Pipeline

We instantiate three pipeline variants and compare their outputs on the same query.

In [ ]:
# Create three pipeline variants
pipe_none = CompressedRAGPipeline(KNOWLEDGE_BASE, embed_texts, index, compress_mode="none")
pipe_ext  = CompressedRAGPipeline(KNOWLEDGE_BASE, embed_texts, index, compress_mode="extractive")
pipe_abs  = CompressedRAGPipeline(KNOWLEDGE_BASE, embed_texts, index, compress_mode="abstractive")

query = "How much does green hydrogen cost compared to grey hydrogen?"

for label, pipe in [("NO COMPRESSION", pipe_none), ("EXTRACTIVE", pipe_ext), ("ABSTRACTIVE", pipe_abs)]:
    print("=" * 70)
    print(f"Pipeline: {label}")
    print("=" * 70)
    answer = pipe.run(query, verbose=True)
    print(f"\nAnswer: {answer}\n")

### Multi-Query Benchmark

Let's run all three pipelines on several queries and compare token usage and compression ratios.

In [ ]:
benchmark_queries = [
    "What is the efficiency of modern solar panels?",
    "How does Denmark generate most of its electricity?",
    "What are the alternatives to lithium-ion batteries for grid storage?",
    "How much energy is needed to produce one kilogram of green hydrogen?",
    "What is the capacity factor of nuclear power plants?",
]

print(f"{'Query':<55} {'Mode':<14} {'Ctx Tokens':>10} {'Ratio':>8}")
print("=" * 95)

for query in benchmark_queries:
    results = pipe_none.retrieve(query)
    chunks = [c for _, _, c in results]

    # Uncompressed
    raw = "\n\n".join(chunks)
    raw_tok = len(tokenizer.encode(raw))

    # Extractive
    ext = "\n\n".join(extractive_compress(c, query, 0.55)[0] for c in chunks)
    ext_tok = len(tokenizer.encode(ext))

    # Abstractive
    abs_parts = [abstractive_compress(c, query)[0] for c in chunks]
    abst = "\n\n".join(p for p in abs_parts if p)
    abs_tok = len(tokenizer.encode(abst))

    q_short = query[:53] + ".." if len(query) > 53 else query
    print(f"{q_short:<55} {'Uncompressed':<14} {raw_tok:>10}")
    print(f"{'':<55} {'Extractive':<14} {ext_tok:>10} {ext_tok/raw_tok:>8.1%}")
    print(f"{'':<55} {'Abstractive':<14} {abs_tok:>10} {abs_tok/raw_tok:>8.1%}")
    print()

## §8 — Latency vs. Quality Trade-off

Compression is not free. Each abstractive compression call requires a full LLM forward pass. Let's measure the actual time cost.

In [ ]:
import time

query = "What is the cost of direct air capture?"
results = retrieve(query, k=3)
chunks = [c for _, _, c in results]

# Time extractive
t0 = time.time()
for c in chunks:
    extractive_compress(c, query, 0.55)
ext_time = time.time() - t0

# Time abstractive
t0 = time.time()
for c in chunks:
    abstractive_compress(c, query)
abs_time = time.time() - t0

# Time generation (raw)
raw_context = "\n\n".join(chunks)
t0 = time.time()
rag_answer(query, raw_context)
gen_raw_time = time.time() - t0

# Time generation (extractive compressed)
ext_context = "\n\n".join(extractive_compress(c, query, 0.55)[0] for c in chunks)
t0 = time.time()
rag_answer(query, ext_context)
gen_ext_time = time.time() - t0

print("Latency Breakdown (3 chunks)")
print(f"{'Stage':<30} {'Time (s)':>10}")
print("-" * 42)
print(f"{'Extractive compression':<30} {ext_time:>10.2f}")
print(f"{'Abstractive compression':<30} {abs_time:>10.2f}")
print(f"{'Generation (raw context)':<30} {gen_raw_time:>10.2f}")
print(f"{'Generation (ext context)':<30} {gen_ext_time:>10.2f}")
print()
print(f"Total — No compression:      {gen_raw_time:.2f}s")
print(f"Total — Extractive:           {ext_time + gen_ext_time:.2f}s")
print(f"Total — Abstractive:          {abs_time + gen_ext_time:.2f}s")

### Handling Edge Cases — Irrelevant Chunks

Sometimes a retrieved chunk has **no relevant information** for the query (especially at lower ranks). Both compressors handle this:

- **Extractive**: falls back to keeping the single best sentence
- **Abstractive**: returns empty string when the LLM outputs `NO_RELEVANT_INFO`

Let's test with a deliberately mismatched query.

In [ ]:
query = "What is the boiling point of water at sea level?"  # not in our KB at all
results = retrieve(query, k=2)

print(f"Query: {query!r}\n")
for rank, (score, idx, chunk) in enumerate(results):
    ext_comp, ext_stats = extractive_compress(chunk, query, threshold=0.55)
    abs_comp, abs_stats = abstractive_compress(chunk, query)
    print(f"--- Chunk {idx} (score={score:.4f}) ---")
    print(f"  Extractive ({ext_stats['kept_sents']}/{ext_stats['original_sents']} sents): {ext_comp[:120]}...")
    print(f"  Abstractive: {abs_comp if abs_comp else '(empty — no relevant info)'}")
    print()

### Token Savings Summary

Let's create a clear summary of token savings across all our benchmark queries.

In [ ]:
print(f"{'Query':<55} {'Raw':>6} {'Ext':>6} {'Abs':>6} {'Ext Save':>9} {'Abs Save':>9}")
print("=" * 100)

total_raw = total_ext = total_abs = 0
for query in benchmark_queries:
    results = retrieve(query, k=3)
    chunks = [c for _, _, c in results]

    raw = "\n\n".join(chunks)
    raw_tok = len(tokenizer.encode(raw))

    ext = "\n\n".join(extractive_compress(c, query, 0.55)[0] for c in chunks)
    ext_tok = len(tokenizer.encode(ext))

    abs_parts = [abstractive_compress(c, query)[0] for c in chunks]
    abst = "\n\n".join(p for p in abs_parts if p)
    abs_tok = len(tokenizer.encode(abst))

    total_raw += raw_tok
    total_ext += ext_tok
    total_abs += abs_tok

    q_short = query[:53] + ".." if len(query) > 53 else query
    print(f"{q_short:<55} {raw_tok:>6} {ext_tok:>6} {abs_tok:>6} "
          f"{raw_tok - ext_tok:>8}↓ {raw_tok - abs_tok:>8}↓")

print("-" * 100)
print(f"{'TOTAL':<55} {total_raw:>6} {total_ext:>6} {total_abs:>6} "
      f"{total_raw - total_ext:>8}↓ {total_raw - total_abs:>8}↓")
print(f"{'AVERAGE RATIO':<55} {'':>6} {total_ext/total_raw:>6.1%} {total_abs/total_raw:>6.1%}")

## §9 — Synthesis: When to Use Contextual Compression

### Decision Framework

| Factor | Favours Compression | Favours Raw Retrieval |
|---|---|---|
| **Context window** | Small window (4k-8k tokens) → compression essential | Large window (128k+) → less critical |
| **Chunk quality** | Chunks are long/noisy → high value from compression | Chunks are short/precise → less benefit |
| **Latency budget** | Tolerant (batch processing) → abstractive OK | Real-time chat → extractive or skip |
| **Accuracy needs** | High stakes (medical, legal) → compression helps focus | Casual queries → raw is fine |
| **Cost constraints** | Cheaper model for generation → save tokens | Unlimited budget → less important |

### Extractive vs. Abstractive

| | Extractive | Abstractive |
|---|---|---|
| **Speed** | ~50ms (embedding only) | ~2-5s per chunk (LLM call) |
| **Compression ratio** | 40-70% retained | 20-40% retained |
| **Hallucination risk** | Zero (preserves original text) | Low but nonzero |
| **Best for** | Time-sensitive apps, high-trust requirements | Maximum compression, synthesis needed |

### Key Takeaways

1. **Compression is most valuable when chunks are long and noisy** — which is the common case in practice.
2. **Extractive compression is nearly free** — the embedding computation is fast and introduces no hallucination risk. It should be the default choice.
3. **Abstractive compression shines for maximum token savings** — useful when context windows are small or when you need to fit many chunks.
4. **The compression step can be parallelised** — process all chunks simultaneously to minimise wall-clock latency.
5. **Compression and reranking are complementary** — rerank first to select the best chunks, then compress them.

### Final Demo — End-to-End Pipeline

One last run showing the full pipeline with verbose output for each stage.

In [ ]:
query = "Compare the cost trends of solar energy and battery storage since 2010."

print("=" * 70)
print(f"QUERY: {query}")
print("=" * 70)

# Step 1: Retrieve
results = retrieve(query, k=3)
print(f"\n[RETRIEVE] Top {len(results)} chunks:")
for rank, (score, idx, chunk) in enumerate(results):
    print(f"  #{rank+1} (chunk {idx}, score={score:.4f}): {chunk[:80]}...")

# Step 2: Compress (extractive)
print(f"\n[COMPRESS — Extractive]")
ext_chunks = []
for _, _, chunk in results:
    comp, stats = extractive_compress(chunk, query, 0.55)
    ext_chunks.append(comp)
    print(f"  {stats['original_chars']}→{stats['compressed_chars']} chars ({stats['compression_ratio']:.1%})")

# Step 3: Generate
ext_context = "\n\n".join(ext_chunks)
print(f"\n[GENERATE] Context: {len(tokenizer.encode(ext_context))} tokens")
answer = rag_answer(query, ext_context)
print(f"\nANSWER:\n{answer}")

# Also show abstractive for comparison
print(f"\n{'='*70}")
print("[COMPRESS — Abstractive]")
abs_chunks = []
for _, _, chunk in results:
    comp, stats = abstractive_compress(chunk, query)
    abs_chunks.append(comp)
    print(f"  {stats['original_chars']}→{stats['compressed_chars']} chars ({stats['compression_ratio']:.1%})")
abs_context = "\n\n".join(c for c in abs_chunks if c)
print(f"\n[GENERATE] Context: {len(tokenizer.encode(abs_context))} tokens")
answer_abs = rag_answer(query, abs_context)
print(f"\nANSWER:\n{answer_abs}")

## Conclusion

Contextual compression is a powerful post-retrieval technique that sits at the heart of production RAG systems. In this notebook we built both flavours from scratch:

- **Extractive compression** using sentence-level embedding similarity — fast, safe, and effective
- **Abstractive compression** using LLM-based query-focused extraction — maximum compression but slower

The key insight is that **retrieval is coarse, but generation needs precision**. Compression bridges that gap by filtering retrieved chunks down to only the query-relevant signal, resulting in:

- **Higher information density** in the prompt (more signal, less noise)
- **Better answer quality** from the generator (less distraction)
- **Lower token costs** for API-based models (fewer input tokens)
- **Faster generation** (shorter context → faster attention)

The trade-off is additional latency for the compression step itself — which is near-zero for extractive and measurable for abstractive. In practice, **extractive compression should be the default** for most RAG systems, with abstractive compression reserved for cases where maximum compression is needed.